# SolarCast results

Plots and tables for the trained models. All logic lives in the `solarcast` package; run `python run_pipeline.py` first so the artifacts and metrics exist.

Short-range models are scored on NSRDB truth. 6/12 h use the Open-Meteo archive as forecast input; 24/48 h use forecasts really issued 1/2 days earlier (Open-Meteo previous runs), which is what live runs get.

In [ ]:
# CONFIGURATION
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from solarcast import config as C

LAT, LON = C.DEFAULT_LAT, C.DEFAULT_LON
paths = C.Paths.for_site(LAT, LON)

In [ ]:
# TEST METRICS (short range)
import pandas as pd
pd.set_option("display.width", 250)

short = pd.read_csv(paths.outputs / "metrics_short_range.csv")
short

In [ ]:
# PLOT: test RMSE by horizon, new ensemble vs baselines
import matplotlib.pyplot as plt
from solarcast import plots

models = ["Ensemble", "Open-Meteo forecast", "Old recipe (Open-Meteo inputs)", "Old recipe (NSRDB inputs)"]
ax = plots.rmse_by_horizon(short, [m for m in models if m in set(short["Model"])])
ax.figure.savefig(paths.outputs / "rmse_by_horizon.png", dpi=150, bbox_inches="tight")

In [ ]:
# REBUILD TEST PREDICTIONS (uses cached data, no download)
from solarcast.models import Bundle
from solarcast.train import prepare_training, test_predictions

site = C.read_site(paths.artifacts)
prep = prepare_training(paths, LAT, LON, site.get("years", C.DEFAULT_YEARS))
bundle = Bundle.load(paths.artifacts)
preds = {h: test_predictions(prep, bundle, h) for h in C.HORIZONS}

In [ ]:
# PLOT: one test week at 6 h and 48 h ahead
for h in (6, 48):
    week = preds[h].loc["2024-09-01":"2024-09-07"]
    series = [s for s in ("Ensemble", "Open-Meteo forecast") if s in week]
    ax = plots.week_timeseries(week, series, f"GHI {h} h ahead, 1-7 Sep 2024 (UTC)")
    ax.figure.savefig(paths.outputs / f"week_h{h}.png", dpi=150, bbox_inches="tight")

In [ ]:
# PLOT: XGBoost feature importance at 6 h
ax = plots.feature_importance(bundle.xgb_models[6])
ax.figure.savefig(paths.outputs / "xgb_feature_importance_h6.png", dpi=150, bbox_inches="tight")

In [ ]:
# TEST METRICS (long range, daily energy in Wh/m^2)
long_path = paths.outputs / "metrics_long_range.csv"
pd.read_csv(long_path) if long_path.exists() else "Long-range models not trained (--skip-long-range)"